# Transform Results Data
1. Read bronze `results` table
1. Keep only the columns required for analytics (Drop `url` column)
1. Standardise column names using snake_case (`constructorId` → `constructor_id`, `driverId` → `driver_id`, `raceName` → `race_name`, `positionText` → `finish_position_text`)
1. Rename columns to make them more meaningful (`date` → `race_date`, `grid` → `grid_position`, `laps` → `completed_laps`, `number` → `car_number`, `position` → `finish_position`)
1. Filter out rows where `season`, `round`, `custructor_id` or `driver_id` is null (business key validation)
1. Remove duplicate records
1. Transform values of column `race_name` to Title Case
1. Write the transformed data to silver `results` table

In [0]:
%run ../00-common/01_Environmnet_config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.results"
silver_table = f"{catalog_name}.{silver_schema}.results"

In [0]:
results_df = spark.read.table(bronze_table)

results_process_df = results_df.drop("url").withColumnsRenamed(
    {
        "constructorId": "constructor_id",
        "driverId": "driver_id",
        "raceName": "race_name",
        "positionText": "finish_position_text",
        "date": "race_date",
        "grid": "grid_position",
        "laps": "completed_laps",
        "number": "car_number",
        "position": "finish_position",
    }
)

In [0]:
results_nxt_process_df = results_process_df.dropna(
    subset=["season", "round", "constructor_id", "driver_id"]
).dropDuplicates()

In [0]:
result_final_df = results_nxt_process_df.withColumn(
    "race_name", F.initcap(F.col("race_name"))
)

In [0]:
(result_final_df.write.format("delta").mode("overwrite").saveAsTable(silver_table))

In [0]:
%sql
select
  *
from
  formula1.silver.results;